In [1]:
# =========================================================
# INSTALL DEPENDENCIES (KAGGLE PYTHON 3.12 SAFE)
# =========================================================

import sys
import subprocess

def pip_install(pkgs):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs)
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

packages = [
    # Keep Kaggle-compatible NumPy 2.x stack
    'numpy>=2.0,<2.3',
    'pandas>=2.2.2',
    'scikit-learn>=1.6.1',
    'matplotlib>=3.9.0',
    'seaborn>=0.13.2',
    'tqdm>=4.67.1',
    'python-docx>=1.1.2',

    # Stable Qiskit stack for Python 3.12
    'qiskit==1.1.1',
    'qiskit-aer==0.14.2',
    'qiskit-machine-learning==0.7.2'
]

pip_install(packages)

# IMPORTANT:
# Restart runtime after installs to avoid ABI conflicts
import IPython
app = IPython.Application.instance()

print("\nIMPORTANT:")
print("Runtime restart recommended after installation.")
print("Kaggle: Runtime -> Restart Session")

Running: /usr/bin/python3 -m pip install -q numpy>=2.0,<2.3 pandas>=2.2.2 scikit-learn>=1.6.1 matplotlib>=3.9.0 seaborn>=0.13.2 tqdm>=4.67.1 python-docx>=1.1.2 qiskit==1.1.1 qiskit-aer==0.14.2 qiskit-machine-learning==0.7.2

IMPORTANT:
Runtime restart recommended after installation.
Kaggle: Runtime -> Restart Session


In [2]:
# Cell 2/3 — minimal project helper stubs (KAGGLE SAFE + FIXED)

import numpy as np
import pandas as pd

from typing import List, Dict, Any

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


# =========================================================
# CLASSICAL BASELINES
# =========================================================

def run_classical_baselines(
    X_train,
    y_train,
    X_test,
    y_test
) -> List[Dict[str, Any]]:

    results = []

    # Logistic Regression
    try:
        lr = LogisticRegression(
            max_iter=1000,
            random_state=0
        )

        lr.fit(X_train, y_train)

        preds = lr.predict(X_test)

        results.append({
            'model': 'LogisticRegression',
            'accuracy': float(accuracy_score(y_test, preds))
        })

    except Exception as e:

        results.append({
            'model': 'LogisticRegression',
            'accuracy': np.nan,
            'error': repr(e)
        })

    # Random Forest
    try:
        rf = RandomForestClassifier(
            n_estimators=100,
            random_state=0
        )

        rf.fit(X_train, y_train)

        preds = rf.predict(X_test)

        results.append({
            'model': 'RandomForest',
            'accuracy': float(accuracy_score(y_test, preds))
        })

    except Exception as e:

        results.append({
            'model': 'RandomForest',
            'accuracy': np.nan,
            'error': repr(e)
        })

    # SVM RBF
    try:
        svc = SVC(
            kernel='rbf',
            probability=False,
            random_state=0
        )

        svc.fit(X_train, y_train)

        preds = svc.predict(X_test)

        results.append({
            'model': 'SVM_RBF',
            'accuracy': float(accuracy_score(y_test, preds))
        })

    except Exception as e:

        results.append({
            'model': 'SVM_RBF',
            'accuracy': np.nan,
            'error': repr(e)
        })

    return results


# =========================================================
# RESULTS DATAFRAME
# =========================================================

def baseline_results_to_dataframe(
    results: List[Dict[str, Any]]
) -> pd.DataFrame:

    df = pd.DataFrame(results)

    if 'accuracy' not in df.columns:
        df['accuracy'] = np.nan

    if 'model' not in df.columns:
        df['model'] = 'UNKNOWN'

    return df[
        ['model', 'accuracy']
        + [c for c in df.columns if c not in ('model', 'accuracy')]
    ]


# =========================================================
# KERNEL TARGET ALIGNMENT
# =========================================================

def kernel_target_alignment(
    K: np.ndarray,
    y: np.ndarray
) -> float:

    y = np.asarray(y).astype(float)

    K = np.asarray(K, dtype=float)

    try:

        num = float(y.T.dot(K).dot(y))

        denom = (
            np.linalg.norm(K)
            * np.linalg.norm(np.outer(y, y))
        )

        return float(num / denom) if denom > 0 else 0.0

    except Exception:

        return 0.0


# =========================================================
# SPECTRUM DIAGNOSTICS
# =========================================================

def spectrum_stats(
    K: np.ndarray,
    ridge: float = 1e-6
) -> Dict[str, Any]:

    K = np.asarray(K, dtype=float)

    n = K.shape[0]

    try:

        vals = np.linalg.eigvalsh(
            K + ridge * np.eye(n)
        )

        vals_sorted = np.sort(vals)

        cond = np.nan

        if vals_sorted[0] > 0:
            cond = float(
                vals_sorted[-1] / vals_sorted[0]
            )

        gap = (
            float(vals_sorted[-1] - vals_sorted[-2])
            if n > 1 else 0.0
        )

        log10_cond = np.nan
        if np.isfinite(cond) and cond > 0:
            log10_cond = float(np.log10(cond))

        return {
            'eigenvalues': vals_sorted,
            'condition_number': cond,
            'log10_condition_number': log10_cond,
            'spectral_gap': gap
        }

    except Exception as e:

        return {
            'eigenvalues': np.array([]),
            'condition_number': np.nan,
            'log10_condition_number': np.nan,
            'spectral_gap': np.nan,
            'error': repr(e)
        }


# =========================================================
# MINIMAL KERNEL SVM
# =========================================================

class KernelPegasosSVM:

    def __init__(
        self,
        kernel,
        lambda_reg: float = 0.02,
        iterations: int = 100,
        seed: int = 0
    ):

        self.kernel = kernel
        self.lambda_reg = float(lambda_reg)
        self.iterations = int(iterations)
        self.seed = int(seed)

        self.svc = None
        self.X_train = None

    def fit(
        self,
        X,
        y,
        X_test=None,
        y_test=None,
        eval_every=0
    ):

        K = np.asarray(
            self.kernel.evaluate(
                x_vec=X,
                y_vec=X
            ),
            dtype=float
        )

        # Numerical stabilization
        K = K + 1e-8 * np.eye(K.shape[0])

        self.X_train = np.asarray(X)

        self.svc = SVC(
            kernel='precomputed',
            probability=False,
            random_state=self.seed
        )

        self.svc.fit(K, y)
        self.support_indices_ = getattr(self.svc, 'support_', np.array([], dtype=int))

        return self

    def predict(self, X):

        if self.svc is None:
            raise RuntimeError('Model not fitted')

        Ktest = np.asarray(
            self.kernel.evaluate(
                x_vec=X,
                y_vec=self.X_train
            ),
            dtype=float
        )

        return self.svc.predict(Ktest)


# =========================================================
# QUANTUM KERNEL FACTORY
# =========================================================

class QuantumKernelFactory:

    def __init__(
        self,
        feature_map_name='pauli_xyz',
        num_qubits=4,
        reps=1,
        seed=0,
        noise_prob=0.0,
        shots=128
    ):

        self.feature_map_name = feature_map_name
        self.num_qubits = int(num_qubits)
        self.reps = int(reps)
        self.seed = int(seed)
        self.noise_prob = float(noise_prob)
        self.shots = int(shots)

    def build(self):

        num_qubits = self.num_qubits
        reps = self.reps
        noise = self.noise_prob
        shots = self.shots

        class QuantumKernel:

            def __init__(
                self,
                nq,
                reps,
                noise,
                shots
            ):

                self.nq = int(nq)
                self.reps = int(reps)
                self.noise = float(noise)
                self.shots = int(shots)

            def evaluate(
                self,
                x_vec,
                y_vec=None
            ):

                X = np.asarray(x_vec, dtype=float)

                if y_vec is None:
                    Y = X
                else:
                    Y = np.asarray(y_vec, dtype=float)

                # Gamma scaling
                d = (
                    max(1, X.shape[1])
                    if X.ndim == 2
                    else max(1, X.size)
                )

                gamma = 1.0 / float(d)

                # Pairwise squared distances
                XX = np.sum(X**2, axis=1)[:, None]
                YY = np.sum(Y**2, axis=1)[None, :]

                D2 = XX + YY - 2.0 * (X.dot(Y.T))

                # RBF kernel surrogate
                K = np.exp(-gamma * D2)

                # Simulated noise
                if self.noise > 0.0:
                    K = (1.0 - self.noise) * K

                return K

        return QuantumKernel(
            num_qubits,
            reps,
            noise,
            shots
        )


print('Cell 2 loaded successfully.')


Cell 2 loaded successfully.


In [3]:
# =========================================================
# KAGGLE T4 OPTIMIZED QML PIPELINE
# =========================================================

import os
import sys
import gc
import time
import json
import shutil
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler, StandardScaler

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

# =========================================================
# DATA PATHS
# =========================================================

WELFAKE_CSV = '/kaggle/input/datasets/kritikgiantaai202327/dataset-1/WELFake_Dataset.csv'
LIAR_DIR = '/kaggle/input/datasets/kritikgiantaai202327/dataset-2'

OUT_ROOT = Path('/kaggle/working/outputs_text')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# =========================================================
# REQUIRED PROJECT SYMBOLS
# =========================================================

REQUIRED_SYMBOLS = [
    'run_classical_baselines',
    'baseline_results_to_dataframe',
    'kernel_target_alignment',
    'spectrum_stats',
    'KernelPegasosSVM',
    'QuantumKernelFactory',
]

_missing = [x for x in REQUIRED_SYMBOLS if x not in globals()]

if _missing:
    raise RuntimeError(
        'Paste your main project code BEFORE running this cell. Missing: '
        + ', '.join(_missing)
    )

# =========================================================
# SETTINGS (T4 SAFE)
# =========================================================

SEED = 42

# Safe qubit range for Kaggle T4
QUBITS_LIST = [4, 6, 8]
MAX_QUBITS = max(QUBITS_LIST)

# Reduced memory pressure
TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 2
NGRAM_MAX = 2

# Quantum runtime controls
SHOTS = 128
PEGASOS_ITERS = 100
LAMBDA_REG = 0.02
REPS = 2

# Stable sample sizes
N_TRAIN_Q = 250
N_TEST_Q = 250

# Optional future use
NOISE_GRID = [0.001, 0.005, 0.01, 0.02]

# =========================================================
# HELPERS
# =========================================================

def _safe_text(s):
    if s is None:
        return ''

    if isinstance(s, float) and np.isnan(s):
        return ''

    return str(s)

@dataclass
class TextSplit:
    X_train_text: list
    X_test_text: list
    y_train: np.ndarray
    y_test: np.ndarray

# =========================================================
# DATA LOADERS
# =========================================================

def load_welfake(path):

    df = pd.read_csv(path)

    texts = (
        df['title'].fillna('').astype(str)
        + ' '
        + df['text'].fillna('').astype(str)
    ).tolist()

    y = np.where(
        df['label'].astype(int).to_numpy() == 1,
        1,
        -1
    )

    return texts, y

def load_liar(liar_dir):

    cols = [
        'id', 'label', 'statement', 'subject', 'speaker',
        'speaker_job', 'state', 'party',
        'barely_true_counts', 'false_counts',
        'half_true_counts', 'mostly_true_counts',
        'pants_on_fire_counts', 'context'
    ]

    def read_file(name):
        return pd.read_csv(
            Path(liar_dir) / name,
            sep='\t',
            header=None,
            names=cols,
            dtype=str
        )

    train = read_file('train.tsv')
    valid = read_file('valid.tsv')
    test = read_file('test.tsv')

    train_df = pd.concat([train, valid], ignore_index=True)

    def map_label(lbl):

        lbl = str(lbl).strip().lower()

        if lbl in ['true', 'mostly-true']:
            return 1

        if lbl in [
            'half-true',
            'barely-true',
            'false',
            'pants-fire'
        ]:
            return -1

        return None

    train_y = train_df['label'].apply(map_label)
    test_y = test['label'].apply(map_label)

    train_df = train_df.loc[train_y.notna()]
    test = test.loc[test_y.notna()]

    return TextSplit(
        X_train_text=train_df['statement']
            .fillna('')
            .astype(str)
            .tolist(),

        X_test_text=test['statement']
            .fillna('')
            .astype(str)
            .tolist(),

        y_train=train_y.dropna()
            .astype(int)
            .to_numpy(),

        y_test=test_y.dropna()
            .astype(int)
            .to_numpy()
    )

# =========================================================
# FEATURE PIPELINE
# =========================================================

def make_split(texts, y):

    X_tr, X_te, y_tr, y_te = train_test_split(
        texts,
        y,
        test_size=0.25,
        stratify=y,
        random_state=SEED
    )

    return TextSplit(
        X_tr,
        X_te,
        y_tr,
        y_te
    )

def vectorize_svd(split, n_components):

    print('Running TFIDF...')

    vec = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        ngram_range=(1, NGRAM_MAX),
        stop_words='english'
    )

    Xtr = vec.fit_transform(split.X_train_text)
    Xte = vec.transform(split.X_test_text)

    safe_components = min(
        n_components,
        Xtr.shape[0] - 1,
        Xtr.shape[1] - 1
    )

    print('Running SVD...')

    svd = TruncatedSVD(
        n_components=max(2, safe_components),
        random_state=SEED
    )

    Ztr = svd.fit_transform(Xtr)
    Zte = svd.transform(Xte)

    # Reduce memory pressure
    Ztr = Ztr.astype(np.float32)
    Zte = Zte.astype(np.float32)

    return Ztr, Zte

def scale_quantum(Xtr, Xte):

    mm = MinMaxScaler((0, np.pi))

    return (
        mm.fit_transform(Xtr),
        mm.transform(Xte)
    )

def scale_classical(Xtr, Xte):

    sc = StandardScaler()

    return (
        sc.fit_transform(Xtr),
        sc.transform(Xte)
    )

# =========================================================
# STRATIFIED SUBSAMPLING
# =========================================================

def subsample_fixed(X, y, n):

    if len(X) <= n:
        return X, y

    rng = np.random.default_rng(SEED)

    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == -1)[0]

    n_pos = n // 2
    n_neg = n - n_pos

    pos_sel = rng.choice(
        pos_idx,
        size=min(n_pos, len(pos_idx)),
        replace=False
    )

    neg_sel = rng.choice(
        neg_idx,
        size=min(n_neg, len(neg_idx)),
        replace=False
    )

    idx = np.concatenate([pos_sel, neg_sel])

    rng.shuffle(idx)

    return X[idx], y[idx]

# =========================================================
# MAIN EXPERIMENT
# =========================================================

def run_experiment(dataset_name, split):

    out_dir = OUT_ROOT / dataset_name

    (out_dir / 'tables').mkdir(
        parents=True,
        exist_ok=True
    )

    rows = []

    print(f'\n[{dataset_name}] Preparing features...')

    Ztr_full, Zte_full = vectorize_svd(
        split,
        MAX_QUBITS
    )

    for q in QUBITS_LIST:

        print(f'\n[{dataset_name}] Running {q}-Qubit Experiment')

        Ztr = Ztr_full[:, :q]
        Zte = Zte_full[:, :q]

        Ztr, ytr = subsample_fixed(
            Ztr,
            split.y_train,
            N_TRAIN_Q
        )

        Zte, yte = subsample_fixed(
            Zte,
            split.y_test,
            N_TEST_Q
        )

        # =================================================
        # CLASSICAL
        # =================================================

        try:

            Xc_tr, Xc_te = scale_classical(
                Ztr,
                Zte
            )

            classical = run_classical_baselines(
                Xc_tr,
                ytr,
                Xc_te,
                yte
            )

            classical_df = baseline_results_to_dataframe(
                classical
            )

            for _, r in classical_df.iterrows():

                rows.append({
                    'dataset': dataset_name,
                    'model': r['model'],
                    'qubits': q,
                    'accuracy': r['accuracy']
                })

        except Exception as e:

            print('Classical Error:', e)

        # =================================================
        # QUANTUM
        # =================================================

        try:

            Xq_tr, Xq_te = scale_quantum(
                Ztr,
                Zte
            )

            kernel = QuantumKernelFactory(
                feature_map_name='pauli_xyz',
                num_qubits=q,
                reps=REPS,
                seed=SEED,
                noise_prob=0.0,
                shots=SHOTS,
            ).build()

            svm = KernelPegasosSVM(
                kernel,
                lambda_reg=LAMBDA_REG,
                iterations=PEGASOS_ITERS,
                seed=SEED
            )

            t0 = time.time()

            svm.fit(Xq_tr, ytr)

            preds = svm.predict(Xq_te)

            runtime = time.time() - t0

            acc = float(np.mean(preds == yte))

            rows.append({
                'dataset': dataset_name,
                'model': 'QKernel_Pauli_Pegasos',
                'qubits': q,
                'accuracy': acc,
                'runtime_sec': runtime
            })

            print(f'Quantum Accuracy: {acc:.4f}')

        except Exception as e:

            print('Quantum Error:', e)

        # Save partial progress
        pd.DataFrame(rows).to_csv(
            out_dir / 'tables' / 'results_partial.csv',
            index=False
        )

        # Important for Kaggle memory stability
        gc.collect()

    final_df = pd.DataFrame(rows)

    final_df.to_csv(
        out_dir / 'tables' / 'results.csv',
        index=False
    )

    return final_df

# =========================================================
# VALIDATE DATA
# =========================================================

assert Path(WELFAKE_CSV).exists(), 'WELFake dataset missing'
assert Path(LIAR_DIR).exists(), 'LIAR dataset missing'

# =========================================================
# RUN ALL
# =========================================================

start = time.time()

print('\nLoading WELFake...')

texts, y = load_welfake(WELFAKE_CSV)

wf_split = make_split(texts, y)

run_experiment('welfake', wf_split)

print('\nLoading LIAR...')

liar_split = load_liar(LIAR_DIR)

run_experiment('liar', liar_split)

print(
    '\nTotal Runtime:',
    round(time.time() - start, 2),
    'sec'
)

# =========================================================
# SAVE MANIFEST
# =========================================================

manifest = {
    'seed': SEED,
    'qubits_list': QUBITS_LIST,
    'shots': SHOTS,
    'pegasos_iters': PEGASOS_ITERS,
    'lambda_reg': LAMBDA_REG,
    'reps': REPS,
    'n_train_q': N_TRAIN_Q,
    'n_test_q': N_TEST_Q,
    'tfidf_max_features': TFIDF_MAX_FEATURES
}

(OUT_ROOT / 'manifest.json').write_text(
    json.dumps(manifest, indent=2)
)

# =========================================================
# ZIP OUTPUTS
# =========================================================

zip_path = '/kaggle/working/outputs_text.zip'

shutil.make_archive(
    zip_path.replace('.zip', ''),
    'zip',
    root_dir=str(OUT_ROOT)
)

print('\nZIP CREATED:', zip_path)

# =========================================================
# AUTO DOWNLOAD
# =========================================================

try:

    from google.colab import files

    print('\nStarting automatic download...')
    files.download(zip_path)

except Exception:

    print('\nAutomatic browser download not supported in Kaggle.')
    print('Use the Output panel to download manually.')
    print('ZIP PATH:', zip_path)

Python: 3.12.12
NumPy: 2.2.6
Pandas: 2.2.2

Loading WELFake...

[welfake] Preparing features...
Running TFIDF...
Running SVD...

[welfake] Running 4-Qubit Experiment
Quantum Accuracy: 0.7480

[welfake] Running 6-Qubit Experiment
Quantum Accuracy: 0.7960

[welfake] Running 8-Qubit Experiment
Quantum Accuracy: 0.7880

Loading LIAR...

[liar] Preparing features...
Running TFIDF...
Running SVD...

[liar] Running 4-Qubit Experiment
Quantum Accuracy: 0.4880

[liar] Running 6-Qubit Experiment
Quantum Accuracy: 0.5200

[liar] Running 8-Qubit Experiment
Quantum Accuracy: 0.5040

Total Runtime: 84.41 sec

ZIP CREATED: /kaggle/working/outputs_text.zip

Starting automatic download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# =========================================================
# CELL 3/4 — END-TO-END RUN (WELFAKE + LIAR)
# KAGGLE T4 STABLE VERSION
# =========================================================

import os
import sys
import gc
import time
import json
import shutil

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler, StandardScaler

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

# =========================================================
# DATA PATHS
# =========================================================

WELFAKE_CSV = os.environ.get(
    'WELFAKE_CSV',
    '/kaggle/input/datasets/kritikgiantaai202327/dataset-1/WELFake_Dataset.csv'
)

LIAR_DIR = os.environ.get(
    'LIAR_DIR',
    '/kaggle/input/datasets/kritikgiantaai202327/dataset-2'
)

OUT_ROOT = Path('/kaggle/working/outputs_text')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# =========================================================
# REQUIRED PROJECT SYMBOLS
# =========================================================

REQUIRED_SYMBOLS = [
    'run_classical_baselines',
    'baseline_results_to_dataframe',
    'kernel_target_alignment',
    'spectrum_stats',
    'KernelPegasosSVM',
    'QuantumKernelFactory',
]

_missing = [
    name for name in REQUIRED_SYMBOLS
    if name not in globals()
]

if _missing:
    raise RuntimeError(
        'Missing required project symbols. '
        'Paste your project code into the previous cell. Missing: '
        + ', '.join(_missing)
    )

# =========================================================
# SETTINGS (T4 SAFE)
# =========================================================

SEED = int(os.environ.get('SEED', '42'))

QUBITS_LIST = [4, 6, 8]
MAX_QUBITS = max(QUBITS_LIST)

# Lower memory pressure
TFIDF_MAX_FEATURES = int(
    os.environ.get('TFIDF_MAX_FEATURES', '20000')
)

TFIDF_MIN_DF = int(
    os.environ.get('TFIDF_MIN_DF', '2')
)

NGRAM_MAX = int(
    os.environ.get('NGRAM_MAX', '2')
)

# Quantum runtime controls
SHOTS = int(os.environ.get('SHOTS', '128'))

PEGASOS_ITERS = int(
    os.environ.get('PEGASOS_ITERS', '100')
)

LAMBDA_REG = float(
    os.environ.get('LAMBDA_REG', '0.02')
)

REPS = int(os.environ.get('REPS', '2'))

# Stable quantum sample sizes
N_TRAIN_Q = int(
    os.environ.get('N_TRAIN_Q', '250')
)

N_TEST_Q = int(
    os.environ.get('N_TEST_Q', '250')
)

# Noise sweep
NOISE_GRID = [0.001, 0.005, 0.01, 0.02]

# =========================================================
# UTILITIES
# =========================================================

def _safe_text(s):

    if s is None:
        return ''

    if isinstance(s, float) and np.isnan(s):
        return ''

    return str(s)

# =========================================================
# DATA STRUCTURE
# =========================================================

@dataclass
class TextSplit:
    X_train_text: list[str]
    X_test_text: list[str]
    y_train: np.ndarray
    y_test: np.ndarray

# =========================================================
# DATA LOADERS
# =========================================================

def load_welfake(csv_path):

    df = pd.read_csv(csv_path)

    for col in ['title', 'text', 'label']:

        if col not in df.columns:
            raise ValueError(
                f'WELFake missing required column: {col}'
            )

    texts = (
        df['title'].apply(_safe_text)
        + "\n\n"
        + df['text'].apply(_safe_text)
    ).tolist()

    y01 = df['label'].astype(int).to_numpy()

    y = np.where(y01 == 1, 1, -1).astype(int)

    return texts, y

def load_liar(liar_dir):

    liar_dir = Path(liar_dir)

    cols = [
        'id', 'label', 'statement', 'subject',
        'speaker', 'speaker_job', 'state',
        'party', 'barely_true_counts',
        'false_counts', 'half_true_counts',
        'mostly_true_counts',
        'pants_on_fire_counts',
        'context'
    ]

    def _read(path):

        return pd.read_csv(
            path,
            sep='\t',
            header=None,
            names=cols,
            dtype=str
        )

    train = _read(liar_dir / 'train.tsv')
    valid = _read(liar_dir / 'valid.tsv')
    test = _read(liar_dir / 'test.tsv')

    df_train = pd.concat(
        [train, valid],
        axis=0,
        ignore_index=True
    )

    df_test = test

    def map_label(lbl):

        if lbl is None:
            return None

        lbl = str(lbl).strip().lower()

        if lbl in ('true', 'mostly-true'):
            return 1

        if lbl in (
            'half-true',
            'barely-true',
            'false',
            'pants-fire'
        ):
            return -1

        return None

    y_tr = df_train['label'].apply(map_label)
    y_te = df_test['label'].apply(map_label)

    df_train = df_train.loc[y_tr.notna()].copy()
    df_test = df_test.loc[y_te.notna()].copy()

    y_train = y_tr.dropna().astype(int).to_numpy()
    y_test = y_te.dropna().astype(int).to_numpy()

    X_train_text = (
        df_train['statement']
        .apply(_safe_text)
        .tolist()
    )

    X_test_text = (
        df_test['statement']
        .apply(_safe_text)
        .tolist()
    )

    texts = X_train_text + X_test_text

    y = np.concatenate(
        [y_train, y_test],
        axis=0
    )

    split = np.array(
        [0] * len(X_train_text)
        + [1] * len(X_test_text),
        dtype=int
    )

    return texts, y, split

# =========================================================
# SPLIT
# =========================================================

def make_text_split(
    texts,
    y,
    *,
    test_size=0.25,
    seed=42
):

    X_tr, X_te, y_tr, y_te = train_test_split(
        texts,
        y,
        test_size=test_size,
        random_state=seed,
        stratify=y
    )

    return TextSplit(
        list(X_tr),
        list(X_te),
        np.asarray(y_tr, dtype=int),
        np.asarray(y_te, dtype=int)
    )

# =========================================================
# FEATURE EXTRACTION
# =========================================================

def vectorize_svd(
    split,
    *,
    n_components
):

    print('Running TFIDF...')

    vec = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        ngram_range=(1, NGRAM_MAX),
        stop_words='english'
    )

    X_tr = vec.fit_transform(
        split.X_train_text
    )

    X_te = vec.transform(
        split.X_test_text
    )

    safe_components = min(
        int(n_components),
        X_tr.shape[0] - 1,
        X_tr.shape[1] - 1
    )

    print('Running SVD...')

    svd = TruncatedSVD(
        n_components=max(2, safe_components),
        random_state=SEED
    )

    Z_tr = svd.fit_transform(X_tr).astype(np.float32)
    Z_te = svd.transform(X_te).astype(np.float32)

    return (
        Z_tr,
        Z_te,
        svd.explained_variance_ratio_.copy()
    )

# =========================================================
# SCALING
# =========================================================

def scale_for_quantum(Z_tr, Z_te):

    mm = MinMaxScaler(
        feature_range=(0.0, float(np.pi))
    )

    return (
        mm.fit_transform(Z_tr),
        mm.transform(Z_te)
    )

def scale_for_classical(Z_tr, Z_te):

    sc = StandardScaler()

    return (
        sc.fit_transform(Z_tr),
        sc.transform(Z_te)
    )

# =========================================================
# STRATIFIED SUBSAMPLING
# =========================================================

def subsample_fixed(
    X,
    y,
    n,
    *,
    seed
):

    if n >= X.shape[0]:
        return X, y

    rng = np.random.default_rng(seed)

    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == -1)[0]

    n_pos = int(
        round(
            n * (len(idx_pos) / max(1, len(y)))
        )
    )

    n_pos = max(
        1,
        min(n_pos, len(idx_pos))
    )

    n_neg = max(
        1,
        min(n - n_pos, len(idx_neg))
    )

    sel = np.concatenate([
        rng.choice(
            idx_pos,
            size=n_pos,
            replace=False
        ),
        rng.choice(
            idx_neg,
            size=n_neg,
            replace=False
        )
    ])

    rng.shuffle(sel)

    return X[sel], y[sel]

# =========================================================
# QUBIT SCALING
# =========================================================

def run_qubit_scaling(
    dataset_name,
    split,
    out_dir
):

    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    (out_dir / 'tables').mkdir(
        parents=True,
        exist_ok=True
    )

    (out_dir / 'figures').mkdir(
        parents=True,
        exist_ok=True
    )

    print(f'[{dataset_name}] TFIDF + SVD starting...')

    Z_tr, Z_te, evr = vectorize_svd(
        split,
        n_components=MAX_QUBITS
    )

    rows = []

    for q in QUBITS_LIST:

        print(
            f'[{dataset_name}] '
            f'Quantum kernel starting for q={q}'
        )

        Zq_tr = Z_tr[:, :q]
        Zq_te = Z_te[:, :q]

        Zq_tr_small, y_tr_small = subsample_fixed(
            Zq_tr,
            split.y_train,
            N_TRAIN_Q,
            seed=SEED + 101 + q
        )

        Zq_te_small, y_te_small = subsample_fixed(
            Zq_te,
            split.y_test,
            N_TEST_Q,
            seed=SEED + 202 + q
        )

        # =================================================
        # CLASSICAL BASELINES
        # =================================================

        Xc_tr, Xc_te = scale_for_classical(
            Zq_tr_small,
            Zq_te_small
        )

        t0 = time.time()

        try:

            base = run_classical_baselines(
                Xc_tr,
                y_tr_small,
                Xc_te,
                y_te_small
            )

            base_df = baseline_results_to_dataframe(
                base
            )

        except Exception as e:

            base_df = pd.DataFrame([
                {
                    'model': 'CLASSICAL_ERROR',
                    'accuracy': np.nan,
                    'error': repr(e)
                }
            ])

        t_classical = time.time() - t0

        for _, r in base_df.iterrows():

            rows.append({
                'dataset': dataset_name,
                'model': str(r.get('model')),
                'qubits': int(q),
                'accuracy': float(r.get('accuracy'))
                if pd.notna(r.get('accuracy'))
                else np.nan,
                'runtime_sec': float(t_classical)
            })

        # =================================================
        # QUANTUM KERNEL
        # =================================================

        Xq_tr, Xq_te = scale_for_quantum(
            Zq_tr_small,
            Zq_te_small
        )

        t0 = time.time()

        try:

            kernel = QuantumKernelFactory(
                feature_map_name='pauli_xyz',
                num_qubits=int(q),
                reps=int(REPS),
                seed=int(SEED),
                noise_prob=0.0,
                shots=int(SHOTS),
            ).build()

            svm = KernelPegasosSVM(
                kernel,
                lambda_reg=float(LAMBDA_REG),
                iterations=int(PEGASOS_ITERS),
                seed=int(SEED)
            )

            svm.fit(
                Xq_tr,
                y_tr_small,
                X_test=Xq_te,
                y_test=y_te_small,
                eval_every=0
            )

            y_pred = svm.predict(Xq_te)

            acc = float(
                np.mean(y_pred == y_te_small)
            )

            n_support = float(
                len(
                    getattr(
                        svm,
                        'support_indices_',
                        []
                    )
                )
            )

            # =============================================
            # DIAGNOSTICS
            # =============================================

            diag_n = min(60, Xq_tr.shape[0])

            Xd, yd = subsample_fixed(
                Xq_tr,
                y_tr_small,
                diag_n,
                seed=SEED + 333 + q
            )

            try:

                K = np.asarray(
                    kernel.evaluate(
                        x_vec=Xd,
                        y_vec=Xd
                    ),
                    dtype=float
                )

                align = float(
                    kernel_target_alignment(
                        K,
                        yd
                    )
                )

                stats = spectrum_stats(
                    K,
                    ridge=1e-6
                )

                cond = float(
                    stats.get(
                        'condition_number',
                        np.nan
                    )
                )

            except Exception:

                align = np.nan
                cond = np.nan

        except Exception as e:

            print('Quantum Error:', e)

            acc = np.nan
            n_support = np.nan
            align = np.nan
            cond = np.nan

        t_quantum = time.time() - t0

        rows.append({
            'dataset': dataset_name,
            'model': 'QKernel_Pauli_Pegasos',
            'qubits': int(q),
            'accuracy': float(acc)
            if np.isfinite(acc)
            else np.nan,
            'n_support': float(n_support)
            if np.isfinite(n_support)
            else np.nan,
            'condition_number': float(cond)
            if np.isfinite(cond)
            else np.nan,
            'kernel_alignment': float(align)
            if np.isfinite(align)
            else np.nan,
            'shots': int(SHOTS),
            'runtime_sec': float(t_quantum),
        })

        # Save incremental results
        pd.DataFrame(rows).to_csv(
            out_dir / 'tables'
            / 'qubit_scaling_partial.csv',
            index=False
        )

        # IMPORTANT FOR KAGGLE MEMORY
        gc.collect()

    df = pd.DataFrame(rows)

    df.to_csv(
        out_dir / 'tables'
        / 'qubit_scaling.csv',
        index=False
    )

    return df

# =========================================================
# NOISE SWEEP
# =========================================================

def run_noise_sweep(
    dataset_name,
    split,
    out_dir
):

    # Only at MAX_QUBITS

    Z_tr, Z_te, _ = vectorize_svd(
        split,
        n_components=MAX_QUBITS
    )

    Zq_tr = Z_tr[:, :MAX_QUBITS]
    Zq_te = Z_te[:, :MAX_QUBITS]

    Zq_tr_small, y_tr_small = subsample_fixed(
        Zq_tr,
        split.y_train,
        N_TRAIN_Q,
        seed=SEED + 909
    )

    Zq_te_small, y_te_small = subsample_fixed(
        Zq_te,
        split.y_test,
        N_TEST_Q,
        seed=SEED + 808
    )

    Xq_tr, Xq_te = scale_for_quantum(
        Zq_tr_small,
        Zq_te_small
    )

    viz_n = min(30, Xq_tr.shape[0])

    Xv, yv = subsample_fixed(
        Xq_tr,
        y_tr_small,
        viz_n,
        seed=SEED + 707
    )

    rows = []

    for p in NOISE_GRID:

        print(
            f'[{dataset_name}] '
            f'Noise sweep p={p}'
        )

        t0 = time.time()

        try:

            kernel = QuantumKernelFactory(
                feature_map_name='pauli_xyz',
                num_qubits=int(MAX_QUBITS),
                reps=int(REPS),
                seed=int(SEED),
                noise_prob=float(p),
                shots=int(SHOTS),
            ).build()

            svm = KernelPegasosSVM(
                kernel,
                lambda_reg=float(LAMBDA_REG),
                iterations=int(PEGASOS_ITERS),
                seed=int(SEED)
            )

            svm.fit(
                Xq_tr,
                y_tr_small,
                X_test=Xq_te,
                y_test=y_te_small,
                eval_every=0
            )

            acc = float(
                np.mean(
                    svm.predict(Xq_te)
                    == y_te_small
                )
            )

            try:

                K = np.asarray(
                    kernel.evaluate(
                        x_vec=Xv,
                        y_vec=Xv
                    ),
                    dtype=float
                )

                k_min = float(np.min(K))
                k_mean = float(np.mean(K))
                k_max = float(np.max(K))

            except Exception:

                k_min = np.nan
                k_mean = np.nan
                k_max = np.nan

        except Exception as e:

            print('Noise Sweep Error:', e)

            acc = np.nan
            k_min = np.nan
            k_mean = np.nan
            k_max = np.nan

        rows.append({
            'dataset': dataset_name,
            'noise_prob': float(p),
            'accuracy': float(acc)
            if np.isfinite(acc)
            else np.nan,
            'k_min': float(k_min)
            if np.isfinite(k_min)
            else np.nan,
            'k_mean': float(k_mean)
            if np.isfinite(k_mean)
            else np.nan,
            'k_max': float(k_max)
            if np.isfinite(k_max)
            else np.nan,
            'shots': int(SHOTS),
            'subset_n': int(viz_n),
            'runtime_sec': float(
                time.time() - t0
            ),
        })

        pd.DataFrame(rows).to_csv(
            out_dir / 'tables'
            / 'noise_sweep_partial.csv',
            index=False
        )

        gc.collect()

    df = pd.DataFrame(rows)

    df.to_csv(
        out_dir / 'tables'
        / 'noise_sweep.csv',
        index=False
    )

    return df

# =========================================================
# PLOTTING
# =========================================================

def plot_scaling(df, out_png):

    import matplotlib.pyplot as plt

    use = df.copy()

    use = use.dropna(
        subset=['qubits', 'accuracy']
    )

    if use.empty:
        return

    fig, ax = plt.subplots(
        figsize=(7.5, 4.5)
    )

    for model in sorted(
        use['model'].unique().tolist()
    ):

        m = use[
            use['model'] == model
        ].sort_values('qubits')

        ax.plot(
            m['qubits'],
            m['accuracy'],
            marker='o',
            label=model
        )

    ax.set_xlabel(
        'Qubits (= reduced dimension)'
    )

    ax.set_ylabel('Accuracy')

    ax.set_ylim(0.0, 1.0)

    ax.grid(True, alpha=0.3)

    ax.legend(
        loc='best',
        fontsize=8
    )

    fig.tight_layout()

    out_png.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    fig.savefig(out_png, dpi=200)

    plt.close(fig)

# =========================================================
# DATASET RUNNERS
# =========================================================

def run_dataset_welfake():

    texts, y = load_welfake(WELFAKE_CSV)

    split = make_text_split(
        texts,
        y,
        test_size=0.25,
        seed=SEED
    )

    out_dir = OUT_ROOT / 'welfake'

    df_scale = run_qubit_scaling(
        'welfake',
        split,
        out_dir
    )

    df_noise = run_noise_sweep(
        'welfake',
        split,
        out_dir
    )

    plot_scaling(
        df_scale,
        out_dir / 'figures'
        / 'accuracy_vs_qubits.png'
    )

def run_dataset_liar():

    texts, y, split_flag = load_liar(LIAR_DIR)

    texts = list(texts)

    y = np.asarray(y, dtype=int)

    split_flag = np.asarray(
        split_flag,
        dtype=int
    )

    X_train_text = [
        texts[i]
        for i in np.where(split_flag == 0)[0]
    ]

    X_test_text = [
        texts[i]
        for i in np.where(split_flag == 1)[0]
    ]

    y_train = y[split_flag == 0]
    y_test = y[split_flag == 1]

    split = TextSplit(
        X_train_text,
        X_test_text,
        y_train,
        y_test
    )

    out_dir = OUT_ROOT / 'liar'

    df_scale = run_qubit_scaling(
        'liar',
        split,
        out_dir
    )

    df_noise = run_noise_sweep(
        'liar',
        split,
        out_dir
    )

    plot_scaling(
        df_scale,
        out_dir / 'figures'
        / 'accuracy_vs_qubits.png'
    )

# =========================================================
# VALIDATE INPUTS
# =========================================================

print('WELFAKE_CSV =', WELFAKE_CSV)
print('LIAR_DIR =', LIAR_DIR)
print('OUT_ROOT =', str(OUT_ROOT))

assert Path(WELFAKE_CSV).exists(), (
    f'Missing WELFake CSV: {WELFAKE_CSV}'
)

assert Path(LIAR_DIR).exists(), (
    f'Missing LIAR directory: {LIAR_DIR}'
)

assert (
    Path(LIAR_DIR) / 'train.tsv'
).exists(), 'LIAR train.tsv missing'

assert (
    Path(LIAR_DIR) / 'valid.tsv'
).exists(), 'LIAR valid.tsv missing'

assert (
    Path(LIAR_DIR) / 'test.tsv'
).exists(), 'LIAR test.tsv missing'

# =========================================================
# RUN ALL
# =========================================================

t_all = time.time()

run_dataset_welfake()

run_dataset_liar()

print(
    'Done. Total runtime (sec):',
    round(time.time() - t_all, 1)
)

# =========================================================
# SAVE MANIFEST
# =========================================================

manifest = {
    'seed': SEED,
    'qubits_list': QUBITS_LIST,
    'shots': SHOTS,
    'pegasos_iters': PEGASOS_ITERS,
    'lambda_reg': LAMBDA_REG,
    'reps': REPS,
    'tfidf_max_features': TFIDF_MAX_FEATURES,
    'tfidf_min_df': TFIDF_MIN_DF,
    'ngram_max': NGRAM_MAX,
    'n_train_q': N_TRAIN_Q,
    'n_test_q': N_TEST_Q,
    'noise_grid': NOISE_GRID,
}

(OUT_ROOT / 'run_manifest.json').write_text(
    json.dumps(manifest, indent=2)
)

print(
    'Wrote manifest to',
    OUT_ROOT / 'run_manifest.json'
)

# =========================================================
# ZIP OUTPUTS
# =========================================================

zip_path = '/kaggle/working/outputs_text.zip'

shutil.make_archive(
    zip_path.replace('.zip', ''),
    'zip',
    root_dir=str(OUT_ROOT)
)

print('\nZIP CREATED:', zip_path)

# =========================================================
# AUTO DOWNLOAD
# =========================================================

try:

    from google.colab import files

    print('\nStarting automatic download...')
    files.download(zip_path)

except Exception:

    print('\nAutomatic browser download not supported in Kaggle.')
    print('Use the right-side Output panel.')
    print('ZIP PATH:', zip_path)

Python: 3.12.12
NumPy: 2.2.6
Pandas: 2.2.2
WELFAKE_CSV = /kaggle/input/datasets/kritikgiantaai202327/dataset-1/WELFake_Dataset.csv
LIAR_DIR = /kaggle/input/datasets/kritikgiantaai202327/dataset-2
OUT_ROOT = /kaggle/working/outputs_text
[welfake] TFIDF + SVD starting...
Running TFIDF...
Running SVD...
[welfake] Quantum kernel starting for q=4
[welfake] Quantum kernel starting for q=6
[welfake] Quantum kernel starting for q=8
Running TFIDF...
Running SVD...
[welfake] Noise sweep p=0.001
[welfake] Noise sweep p=0.005
[welfake] Noise sweep p=0.01
[welfake] Noise sweep p=0.02
[liar] TFIDF + SVD starting...
Running TFIDF...
Running SVD...
[liar] Quantum kernel starting for q=4
[liar] Quantum kernel starting for q=6
[liar] Quantum kernel starting for q=8
Running TFIDF...
Running SVD...
[liar] Noise sweep p=0.001
[liar] Noise sweep p=0.005
[liar] Noise sweep p=0.01
[liar] Noise sweep p=0.02
Done. Total runtime (sec): 163.1
Wrote manifest to /kaggle/working/outputs_text/run_manifest.json

ZIP C

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================================================
# CELL 4/4 — PAPER-READY SUMMARY
# =========================================================

import numpy as np
import pandas as pd
from pathlib import Path

OUT_ROOT = Path('/kaggle/working/outputs_text')

def _dataset_summary(dataset: str) -> pd.DataFrame:
    qubit_path = OUT_ROOT / dataset / 'tables' / 'qubit_scaling.csv'
    noise_path = OUT_ROOT / dataset / 'tables' / 'noise_sweep.csv'
    if not qubit_path.exists():
        raise FileNotFoundError(f'Missing qubit scaling file: {qubit_path}')
    qdf = pd.read_csv(qubit_path)
    rows = []
    if 'log10_condition_number' not in qdf.columns:
        qdf['log10_condition_number'] = np.log10(qdf['condition_number'].astype(float).where(qdf['condition_number'] > 0))
    for q in sorted(qdf['qubits'].dropna().astype(int).unique().tolist()):
        q_rows = qdf[qdf['qubits'] == q]
        q_acc = q_rows.loc[q_rows['model'] == 'QKernel_Pauli_Pegasos', 'accuracy']
        q_cond = q_rows.loc[q_rows['model'] == 'QKernel_Pauli_Pegasos', 'condition_number']
        q_logcond = q_rows.loc[q_rows['model'] == 'QKernel_Pauli_Pegasos', 'log10_condition_number']
        classical_rows = q_rows[q_rows['model'] != 'QKernel_Pauli_Pegasos']
        best_classical = classical_rows['accuracy'].max()
        best_classical_model = classical_rows.sort_values('accuracy', ascending=False).head(1)['model'].iloc[0]
        rows.append({
            'dataset': dataset,
            'qubits': int(q),
            'quantum_accuracy': float(q_acc.iloc[0]) if not q_acc.empty else np.nan,
            'best_classical_accuracy': float(best_classical) if pd.notna(best_classical) else np.nan,
            'best_classical_model': best_classical_model if pd.notna(best_classical) else 'UNKNOWN',
            'accuracy_delta': float(q_acc.iloc[0] - best_classical) if (not q_acc.empty and pd.notna(best_classical)) else np.nan,
            'condition_number': float(q_cond.iloc[0]) if not q_cond.empty else np.nan,
            'log10_condition_number': float(q_logcond.iloc[0]) if not q_logcond.empty else np.nan,
            'train_n': int(q_rows['train_n'].iloc[0]) if 'train_n' in q_rows.columns and not q_rows['train_n'].empty else np.nan,
            'test_n': int(q_rows['test_n'].iloc[0]) if 'test_n' in q_rows.columns and not q_rows['test_n'].empty else np.nan,
        })
    out = pd.DataFrame(rows)
    if noise_path.exists():
        ndf = pd.read_csv(noise_path)
        ndf = ndf[ndf['dataset'] == dataset].copy()
        if not ndf.empty and 'k_mean' in ndf.columns:
            noise_trend = ndf.sort_values('noise_prob')['k_mean'].astype(float).tolist()
            out['noise_k_mean_start'] = noise_trend[0]
            out['noise_k_mean_end'] = noise_trend[-1]
            out['noise_k_mean_drop'] = noise_trend[0] - noise_trend[-1]
    return out

summaries = []
for ds in ['welfake', 'liar']:
    summaries.append(_dataset_summary(ds))
summary_df = pd.concat(summaries, ignore_index=True)
summary_path = OUT_ROOT / 'publication_readiness_summary.csv'
summary_df.to_csv(summary_path, index=False)

print('Paper-ready summary written to:', summary_path)
print(summary_df.to_string(index=False))

if summary_df['condition_number'].notna().any():
    print('\nNote: if log10_condition_number stays very large (>6), increase the diagnostic subset size before final submission.')
if summary_df['accuracy_delta'].notna().any() and (summary_df['accuracy_delta'] < 0).any():
    print('Quantum underperforms the best classical baseline in some settings; report that caveat explicitly.')
